# Notebook 2 — Consolidação das DFPs na Camada Silver

**Objetivo:** Extrair os 7 tipos de demonstrativo da bronze, normalizar escala (MIL → UNIDADE), deduplicar versões, identificar contas folha (IS_LEAF) e consolidar em uma única tabela Silver.  
**Input:** `layer_01_bronze.dfp_cia_aberta_bpa/bpp/dre/dra/dmpl/dfc_mi`  
**Output:** `layer_02_silver.n0_dfp_cia_aberta`

## 1. Importando Bibliotecas

In [2]:
import pandas as pd
import datetime
import logging
import os
from sqlalchemy import create_engine, text, inspect
from urllib.parse import quote_plus
from dotenv import load_dotenv

In [3]:
# Carrega variáveis de ambiente
load_dotenv()

True

In [4]:
# Definindo a função para criar a engine do banco de dados
def create_db_engine():
        user = quote_plus(os.getenv('DB_USER', 'postgres'))
        password = quote_plus(os.getenv('DB_PASS', 'password'))
        host = os.getenv('DB_HOST', 'localhost')
        port = os.getenv('DB_PORT', '5432')
        dbname = os.getenv('DB_NAME', 'data_lake')
        
        connection_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
        return create_engine(connection_str)

In [5]:
# Efetivamente criando a engine
engine = create_db_engine()

## 2. Query Agnóstica — template reutilizado para todos os 7 demonstrativos

> A query recebe `{tabela_dfp}` e `{filtros_adicionais}` como parâmetros via `.format()`,  
> permitindo processar BPA, BPP, DRE, DRA, DMPL, DFC e DVA com o mesmo código.  
> Três decisões de design estão embutidas aqui: deduplicação por VERSÃO, normalização de escala e exceção LPA.

In [6]:
query_agnostica = '''

WITH base_filtrada AS (
    SELECT *
    FROM layer_01_bronze.cad_cia_aberta
    WHERE "SIT" = 'ATIVO'
      AND "TP_MERC" = 'BOLSA'
      AND "SIT_EMISSOR" = 'FASE OPERACIONAL'
      AND "SETOR_ATIV" NOT LIKE '%Emp. Adm. Part%'
      AND "SETOR_ATIV" NOT LIKE '%Banc%'
      AND "SETOR_ATIV" NOT LIKE '%Segurad%'
      AND "SETOR_ATIV" NOT LIKE '%Financeira%'
      AND "SETOR_ATIV" NOT LIKE '%Securitiz%'
),
dados_financeiros AS (
    SELECT 
        t1."CNPJ_CIA",
        t1."DT_REFER",
        t1."DT_REFER"::date as "DT_REFER_TRATADO",
        EXTRACT(year from t1."DT_REFER"::date) as "DT_REFER_ANO",
        t1."VERSAO",
        t1."DENOM_CIA",
        t1."CD_CVM",
        t1."GRUPO_DFP",
        t1."MOEDA",
        t1."ESCALA_MOEDA",
        t1."ORDEM_EXERC",
        t1."DT_FIM_EXERC",
        t1."DT_FIM_EXERC"::date as "DT_FIM_EXERC_TRATADO",
        EXTRACT(year from t1."DT_FIM_EXERC"::date) as "DT_FIM_EXERC_ANO",
        t1."CD_CONTA",
        LENGTH(REPLACE(t1."CD_CONTA", '.','')) as "CD_CONTA_QTD_DIGITOS",
        t1."DS_CONTA",
        t1."VL_CONTA",
        case
            -- EXCEÇÃO DRE: Lucro por Ação (3.99) é sempre unitário
            -- Essa regra deve vir PRIMEIRO para prevalecer sobre a escala
            when t1."CD_CONTA" LIKE '3.99%'
                then t1."VL_CONTA"::numeric
            
            -- REGRA GERAL: Se for MIL, trunca as casas decimais do valor original ANTES de multiplicar
            when TRIM(UPPER(t1."ESCALA_MOEDA")) like '%MIL%'
                then TRUNC(t1."VL_CONTA"::numeric) * 1000 
            
            -- REGRA GERAL: Se for UNIDADE, mantém o valor exato
            else t1."VL_CONTA"::numeric
        end as "VL_CONTA_TRATADO",
        t1."ST_CONTA_FIXA",
        
        DENSE_RANK() OVER (
            PARTITION BY t1."CNPJ_CIA", t1."DT_REFER"
            ORDER BY t1."VERSAO" DESC
        ) as rn
        
    FROM layer_01_bronze.{tabela_dfp} as t1 
    WHERE t1."ORDEM_EXERC" = 'ÚLTIMO'
      {filtros_adicionais} 
)
SELECT 
    d."CNPJ_CIA",
    d."DT_REFER",
    d."DT_REFER_TRATADO",
    d."DT_REFER_ANO",
    d."VERSAO",
    d."DENOM_CIA",
    d."CD_CVM",
    d."GRUPO_DFP",
    case
        when UPPER(TRIM(d."GRUPO_DFP")) like '%BALANÇO PATRIMONIAL%'
            then 'BP'
        when UPPER(TRIM(d."GRUPO_DFP")) like '%MUTAÇÕES DO PATRIMÔNIO LÍQUIDO%'
            then 'DMPL'
        when UPPER(TRIM(d."GRUPO_DFP")) like '%RESULTADO ABRANGENTE%'
            then 'DRA'
        when UPPER(TRIM(d."GRUPO_DFP")) like '%VALOR ADICIONADO%'
            then 'DVA'
        when UPPER(TRIM(d."GRUPO_DFP")) like '%FLUXO DE CAIXA%'
            then 'DFC'
        when UPPER(TRIM(d."GRUPO_DFP")) like '%DEMONSTRAÇÃO DO RESULTADO%'
            then 'DRE'
        else 'VALIDAR'
    end as "GRUPO_DFP_TRATADO",
    d."MOEDA",
    d."ESCALA_MOEDA",
    d."ORDEM_EXERC",
    d."DT_FIM_EXERC",
    d."DT_FIM_EXERC_TRATADO",
    d."DT_FIM_EXERC_ANO",
    d."CD_CONTA",
    d."CD_CONTA_QTD_DIGITOS",
    d."DS_CONTA",
    d."VL_CONTA",
    d."VL_CONTA_TRATADO",
    d."ST_CONTA_FIXA"
FROM dados_financeiros d
INNER JOIN base_filtrada b 
        ON d."CNPJ_CIA" = b."CNPJ_CIA"
WHERE d.rn = 1 
      and d."VL_CONTA_TRATADO" != 0
ORDER BY d."CNPJ_CIA", d."DT_REFER" ASC, d."CD_CONTA" ASC;

'''

## 3. Funções de Processamento — IS_LEAF e orquestração do pipeline

> `processar_leafs_pandas_rapido`: identifica contas folha via teoria de conjuntos —  
> uma conta é IS_LEAF se nenhuma outra conta no mesmo contexto empresa/período a tem como prefixo.  
> `consolidar_e_carregar_dfp`: itera sobre as tabelas, chama a query, aplica IS_LEAF e escreve na Silver.

In [7]:
def processar_leafs_pandas_rapido(df_input: pd.DataFrame) -> pd.DataFrame:
    """
    Identifica contas Leaf (Folhas) usando Teoria dos Conjuntos Expandida.
    Regra: Se uma conta é ANCESTRAL (Pai, Avô, Bisavô...) de qualquer conta ativa,
    ela NÃO é folha.
    """
    df = df_input.copy()

    # 1. Garantia de Tipagem
    if 'VL_CONTA_TRATADO' not in df.columns:
        df['VL_CONTA_TRATADO'] = pd.to_numeric(df['VL_CONTA'], errors='coerce').fillna(0)

    # 2. Função que gera TODOS os ancestrais de uma conta
    # Ex: '1.01.02' -> Retorna ['1', '1.01']
    def get_all_ancestors(s):
        s = str(s)
        if '.' not in s:
            return []
        
        parts = s.split('.')
        # Gera todas as combinações de prefixos progressivos
        # Se parts = ['2', '01', '02', '01', '01']
        # Gera: '2', '2.01', '2.01.02', '2.01.02.01'
        ancestors = []
        current = parts[0]
        ancestors.append(current) # Adiciona o nível 1 (ex: '2')
        
        for part in parts[1:-1]: # Vai até o penúltimo, pois o último é ele mesmo
            current += '.' + part
            ancestors.append(current)
            
        return ancestors

    # 3. Filtrar ativos e gerar a "Nuvem de Ancestrais"
    mask_saldo = df['VL_CONTA_TRATADO'] != 0
    df_ativos = df[mask_saldo].copy()
    
    # Cria uma lista de ancestrais para cada linha
    df_ativos['ANCESTRAIS'] = df_ativos['CD_CONTA'].apply(get_all_ancestors)
    
    # "Explode" a lista para ter uma linha por ancestral
    # Isso cria uma tabela vertical com TODOS os códigos que são pais de alguém
    df_ancestrais_flat = df_ativos.explode('ANCESTRAIS').dropna(subset=['ANCESTRAIS'])
    
    # 4. Criar o Conjunto de Pais (Contexto Único)
    # Quem estiver neste conjunto é pai/avô de alguém e NÃO PODE ser leaf
    conjunto_pais = set(zip(
        df_ancestrais_flat['CNPJ_CIA'], 
        df_ancestrais_flat['DT_REFER'], 
        df_ancestrais_flat['VERSAO'], 
        df_ancestrais_flat['ANCESTRAIS']
    ))

    # 5. Validação Final
    identidade_linha = zip(
        df['CNPJ_CIA'], 
        df['DT_REFER'], 
        df['VERSAO'], 
        df['CD_CONTA'].astype(str)
    )

    # Se eu (meu código) estou na lista de ancestrais de alguém, não sou folha.
    df['IS_LEAF'] = [ident not in conjunto_pais for ident in identidade_linha]

    return df

In [8]:
# Configuração básica de log
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("CVMLakeBuilder")

def consolidar_e_carregar_dfp(
    engine,
    lista_tabelas: list[str],
    query_template: str,
    target_schema: str,
    target_table: str,
    if_exists: str = 'replace'
) -> None:
    """
    Itera sobre uma lista de tabelas DFP (Bronze), executa uma query padronizada,
    consolida os resultados em um único DataFrame e escreve na camada Silver.
    """
    dfs_para_concatenar = []
    
    logger.info(f"Iniciando processo de consolidação para {len(lista_tabelas)} tabelas.")

    try:
        # 1. Extração e Consolidação em Memória
        with engine.connect() as conn:
            for tabela in lista_tabelas:
                try:
                    logger.info(f"Processando tabela: {tabela}...")
                    
                    # --- LÓGICA NOVA: FILTRO DINÂMICO PARA DMPL ---
                    filtros_extras = ""
                    
                    if 'dmpl' in tabela.lower():
                        # Filtra apenas a coluna que representa o Total do PL Consolidado
                        # Usamos ILIKE e % para garantir que pegue variações de acentuação ou espaços
                        filtros_extras = '''AND t1."COLUNA_DF" ILIKE 'Patrim%nio%L%quido%Consolidado'
                        '''
                    
                    # Formata a query injetando o nome da tabela E o filtro (se houver)
                    sql = query_template.format(
                        tabela_dfp=tabela,
                        filtros_adicionais=filtros_extras
                    )
                    # -----------------------------------------------
                    
                    # Executa a leitura
                    df_temp = pd.read_sql(text(sql), con=conn)
                    
                    if not df_temp.empty:
                        df_temp['_origem_tabela'] = f'layer_01_bronze.{tabela}'
                        dfs_para_concatenar.append(df_temp)
                    else:
                        logger.warning(f"A tabela {tabela} não retornou dados.")
                        
                except Exception as e:
                    logger.error(f"Erro ao processar tabela {tabela}: {e}")
                    raise e

        # 2. Concatenação
        if not dfs_para_concatenar:
            logger.warning("Nenhum dado foi coletado. O processo será abortado.")
            return

        logger.info("Concatenando DataFrames...")
        df_consolidado = pd.concat(dfs_para_concatenar, ignore_index=True)

        # ======================================================================
        # INSERÇÃO DA LÓGICA IS_LEAF
        # ======================================================================
        logger.info("Aplicando regra de hierarquia (IS_LEAF) para todos os demonstrativos...")
        
        df_final = processar_leafs_pandas_rapido(df_consolidado)
        
        logger.info(f"Hierarquia processada. Total de Leafs identificados: {df_final['IS_LEAF'].sum()}")
        # ======================================================================


        # 3. Escrita na Camada Silver
        logger.info(f"Escrevendo dados em {target_schema}.{target_table} (Modo: {if_exists})...")
        
        df_final.to_sql(
            name=target_table,
            schema=target_schema,
            con=engine,
            if_exists=if_exists,
            index=False,
            chunksize=10000, 
            method='multi' 
        )
        
        logger.info("Processo finalizado com sucesso.")

    except Exception as e:
        logger.critical(f"Falha crítica no processo de consolidação: {e}")
        raise e

## 4. Execução — processando os 7 demonstrativos

In [9]:
lista_de_tabelas_dfp = [
    'dfp_cia_aberta_bpa_con',
    'dfp_cia_aberta_bpp_con',
    'dfp_cia_aberta_dre_con',
    'dfp_cia_aberta_dra_con',
    'dfp_cia_aberta_dmpl_con',
    'dfp_cia_aberta_dfc_mi_con',
    'dfp_cia_aberta_dva_con'
]

# Executando a função
consolidar_e_carregar_dfp(
    engine=engine,
    lista_tabelas=lista_de_tabelas_dfp,
    query_template=query_agnostica,  # A sua query com .format(tabela_dfp=...)
    target_schema='layer_02_silver',
    target_table='n0_dfp_cia_aberta',
    if_exists='replace' # Use 'replace' para recriar a tabela consolidada ou 'append' para adicionar
)

2026-03-27 15:55:49,294 - INFO - Iniciando processo de consolidação para 7 tabelas.
2026-03-27 15:55:49,387 - INFO - Processando tabela: dfp_cia_aberta_bpa_con...
2026-03-27 15:55:52,336 - INFO - Processando tabela: dfp_cia_aberta_bpp_con...
2026-03-27 15:55:58,297 - INFO - Processando tabela: dfp_cia_aberta_dre_con...
2026-03-27 15:56:03,839 - INFO - Processando tabela: dfp_cia_aberta_dra_con...
2026-03-27 15:56:04,852 - INFO - Processando tabela: dfp_cia_aberta_dmpl_con...
2026-03-27 15:56:09,970 - INFO - Processando tabela: dfp_cia_aberta_dfc_mi_con...
2026-03-27 15:56:14,934 - INFO - Processando tabela: dfp_cia_aberta_dva_con...
2026-03-27 15:56:20,672 - INFO - Concatenando DataFrames...
2026-03-27 15:56:20,753 - INFO - Aplicando regra de hierarquia (IS_LEAF) para todos os demonstrativos...
2026-03-27 15:56:24,298 - INFO - Hierarquia processada. Total de Leafs identificados: 297312
2026-03-27 15:56:24,298 - INFO - Escrevendo dados em layer_02_silver.n0_dfp_cia_aberta (Modo: replace